# Hypothetical Outcome Plot (HOP)

**Family:** Uncertainty &nbsp;|&nbsp; **Library:** plotly

> *Uncertainty:* Show what is not known (intervals, probabilities and possible outcomes) so readers do not over-trust a single number.

## Design purpose

Animates uncertainty instead of summarizing it. Each frame shows one plausible outcome drawn from the distribution, so viewers experience how often one option beats another rather than reading overlapping error bars. Hullman et al. (2015) found HOPs improve judgements of the probability that A > B.

## When to use

- Answer "how often is A better than B?" when the uncertainty intervals overlap
- Communicate model uncertainty to non-experts in an interactive or presentation setting
- Show that a "winner" by average is not a guaranteed winner

## Avoid when

- The output is printed or static (it needs animation; use gradient intervals or half-eye instead)
- Readers need precise interval values
- The animation would run too fast to follow (under ~300 ms per frame)

## Alternatives

- Half-eye plot (static)
- Quantile dotplot
- Bar chart with intervals plus a stated probability that A > B

## Design tips

- Hold axes fixed across frames so only the data moves
- Run at 2–3 frames per second, with play/pause and a slider
- Also state the summary probability in text for those who cannot watch it

## Example

Days to recovery under two treatments, with uncertain averages (Treatment A ~ N(10, 1.6) and Treatment B ~ N(11.5, 1.6) days). Each frame is one draw; synthetic data.

### Setup

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
           "#e87ba4", "#008300", "#6250d6", "#e34948"]
pio.templates["vizlib"] = go.layout.Template(layout=dict(
    colorway=PALETTE, font=dict(family="Helvetica, Arial, sans-serif", size=13, color="#0b0b0b"),
    paper_bgcolor="#fcfcfb", plot_bgcolor="#fcfcfb", title=dict(x=0.02, xanchor="left"),
    xaxis=dict(gridcolor="#e6e5e1", linecolor="#c9c8c3", zeroline=False),
    yaxis=dict(gridcolor="#e6e5e1", linecolor="#c9c8c3", zeroline=False),
))
pio.templates.default = "plotly_white+vizlib"
pio.renderers.default = "plotly_mimetype+notebook_connected"

### Data

In [2]:
rng = np.random.default_rng(4)
n_frames = 40
draws = pd.DataFrame({
    "A": rng.normal(10, 1.6, n_frames),
    "B": rng.normal(11.5, 1.6, n_frames),
})
p_b_slower = (draws["B"] > draws["A"]).mean()
print(f"B slower than A in {p_b_slower:.0%} of draws")
draws.head().round(1)

B slower than A in 75% of draws


,A,B
0,9.0,12.7
1,9.7,12.9
2,12.7,10.4
3,11.1,10.5
4,7.4,10.2


### Plot

In [3]:
labels = ["Treatment A", "Treatment B"]
colors = [PALETTE[0], PALETTE[1]]

def frame_data(i):
    vals = draws.iloc[i][["A", "B"]].values
    return [go.Bar(x=labels, y=vals, marker_color=colors, width=0.5,
                   text=[f"{v:.1f} d" for v in vals], textposition="outside",
                   hovertemplate="%{x}: %{y:.1f} days<extra></extra>")]

def frame_note(i):
    a, b = draws.iloc[i][["A", "B"]]
    who = "A recovers faster" if a < b else "B recovers faster"
    return [dict(text=f"Outcome {i + 1} of {n_frames}: {who}", x=0.5, y=1.02, xref="paper",
                 yref="paper", showarrow=False, font=dict(size=14, color="#52514e"))]

frames = [go.Frame(data=frame_data(i), name=str(i), layout=dict(annotations=frame_note(i)))
          for i in range(n_frames)]
fig = go.Figure(data=frame_data(0), frames=frames)
fig.update_layout(
    title=f"Days to recovery: one plausible outcome per frame (B slower in {p_b_slower:.0%} of draws)",
    yaxis=dict(range=[0, 17], title="Days to recovery"),
    annotations=frame_note(0), showlegend=False, height=520,
    updatemenus=[dict(type="buttons", x=0, y=-0.12, xanchor="left", direction="left", buttons=[
        dict(label="▶ Play", method="animate",
             args=[None, dict(frame=dict(duration=450, redraw=True), transition=dict(duration=0),
                              fromcurrent=True)]),
        dict(label="❚❚ Pause", method="animate",
             args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")]),
    ])],
    sliders=[dict(x=0.18, y=-0.08, len=0.8, currentvalue=dict(prefix="Outcome "),
                  steps=[dict(label=str(i + 1), method="animate",
                              args=[[str(i)], dict(frame=dict(duration=0, redraw=True), mode="immediate")])
                         for i in range(n_frames)])],
)
fig.show()

## How to read it

Press play. B usually takes longer, but in roughly a quarter of the frames A is the slower one. The averages differ, yet the outcome is far from certain, which a pair of overlapping error bars rarely makes clear.

---
Standalone script: [`hypothetical_outcomes.py`](hypothetical_outcomes.py) &nbsp;·&nbsp; [Catalogue](../../README.md) &nbsp;·&nbsp; [Gallery](../../gallery/index.html)